# Historical drought exposure map

Pick a **state**, **crop**, **drought definition** and **map measure**; the county map updates.

**What this map shows.** It describes what *already happened* in 2000–2025. It is **historical exposure**, not a forecast and not a risk estimate ("at risk").

**Drought definitions.** `max_drought_severity` in the master dataset is the worst U.S. Drought Monitor category reached in any week of the growing season (more than 1% of the county affected). The scale is offset by 1 from USDM's own labels:

| Definition | `max_drought_severity` | USDM label |
|---|---|---|
| D1+ moderate or worse | ≥ 2 | D1–D4 |
| D2+ severe or worse | ≥ 3 | D2–D4 |
| D3+ extreme or worse | ≥ 4 | D3–D4 |
| D4 exceptional | = 5 | D4 |

A county-year counts as a **drought year** when its season maximum reaches the chosen level.

**Map measures.**
- *Drought exposure*: number of drought years per county (hover shows the share of observed years).
- *Drought-year losses*: mean `yield_anomaly_pct` over that county's drought years that have a reported yield. Negative = below the county's own trend. Counties with no drought year that has a yield are left blank (blank is not zero).


In [1]:
import sys
sys.path.insert(0, "..")   # this notebook lives in notebooks/, src/ is one level up

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

from src.data_loader import load_master, load_counties

df = load_master()
counties = load_counties()   # GeoJSON, feature id = fips
print(f"{len(df):,} rows loaded")

17,056 rows loaded


In [2]:
# Dropdown options: label shown -> value used in the data
STATES = {
    "Iowa": "IA",
    "Nebraska": "NE",
    "California": "CA",
    "Delaware": "DE",
}
CROPS = {
    "Corn (grain)": "CORN",
    "Soybeans": "SOYBEANS",
    "Winter wheat": "WHEAT",
    "Grain sorghum": "SORGHUM",
}
# max_drought_severity is offset by 1 from USDM labels: 2 = D1, 3 = D2, 4 = D3, 5 = D4
DROUGHT_DEFS = {
    "D1+ moderate or worse (severity ≥ 2)": 2,
    "D2+ severe or worse (severity ≥ 3)": 3,
    "D3+ extreme or worse (severity ≥ 4)": 4,
    "D4 exceptional (severity = 5)": 5,
}
MEASURES = {
    "Drought exposure (number of drought years)": "exposure",
    "Drought-year losses (mean yield anomaly)": "loss",
}

In [3]:
def county_table(state, crop, threshold):
    """One row per county for the chosen state, crop and drought definition."""
    sub = df[(df["state_alpha"] == state) & (df["crop"] == crop)].copy()

    # Drop rows without drought data (e.g. wheat 2000, whose season starts before the
    # Drought Monitor archive does). NaN is not "no drought".
    sub = sub[sub["max_drought_severity"].notna()]
    sub["is_drought"] = sub["max_drought_severity"] >= threshold

    # Exposure: how many seasons reached the chosen drought level
    out = (
        sub.groupby(["fips", "county_name"], as_index=False)
        .agg(years_observed=("year", "nunique"),
             drought_years=("is_drought", "sum"))
    )
    out["pct_years_in_drought"] = 100 * out["drought_years"] / out["years_observed"]

    # Losses: yield anomaly in those drought years, only where a yield (and trend) exists
    dy = sub[sub["is_drought"] & sub["yield_anomaly_pct"].notna()]
    loss = (
        dy.groupby("fips", as_index=False)
        .agg(mean_yield_anomaly_pct=("yield_anomaly_pct", "mean"),
             worst_yield_anomaly_pct=("yield_anomaly_pct", "min"),
             drought_years_with_yield=("year", "nunique"))
    )
    return out.merge(loss, on="fips", how="left")

In [4]:
def make_map(state_label, crop_label, def_label, measure):
    state, crop, threshold = STATES[state_label], CROPS[crop_label], DROUGHT_DEFS[def_label]
    tbl = county_table(state, crop, threshold)

    if tbl.empty:
        return None, f"No data for {crop_label} in {state_label} (that crop is not grown there in this dataset)."

    base_hover = {
        "fips": False,
        "drought_years": True,
        "years_observed": True,
        "pct_years_in_drought": ":.0f",
        "drought_years_with_yield": True,
        "mean_yield_anomaly_pct": ":.1f",
    }
    labels = {
        "drought_years": "Drought years",
        "years_observed": "Years observed",
        "pct_years_in_drought": "% of years in drought",
        "drought_years_with_yield": "Drought years with yield",
        "mean_yield_anomaly_pct": "Mean yield anomaly (%)",
    }
    title_core = f"{state_label} {crop_label}: {def_label}, 2000–2025"

    if measure == "exposure":
        plot_df = tbl
        fig = px.choropleth(
            plot_df, geojson=counties, locations="fips",
            color="drought_years", color_continuous_scale="YlOrRd",
            range_color=(0, max(1, int(plot_df["drought_years"].max()))),
            hover_name="county_name", hover_data=base_hover, labels=labels,
            title=f"Historical drought exposure<br><sup>{title_core}</sup>",
        )
        fig.update_layout(coloraxis_colorbar_title="Drought<br>years")
        note = f"{len(plot_df)} counties shown. Counts seasons that reached the chosen drought level."
    else:
        plot_df = tbl.dropna(subset=["mean_yield_anomaly_pct"])
        if plot_df.empty:
            return None, ("No county has a drought year with a reported yield under this definition. "
                          "Try a milder drought definition.")
        fig = px.choropleth(
            plot_df, geojson=counties, locations="fips",
            color="mean_yield_anomaly_pct", color_continuous_scale="RdYlGn",
            range_color=(-40, 40),
            hover_name="county_name", hover_data=base_hover, labels=labels,
            title=f"Historical drought-year losses<br><sup>{title_core}</sup>",
        )
        fig.update_layout(coloraxis_colorbar_title="Mean yield<br>anomaly (%)")
        thin = int((plot_df["drought_years_with_yield"] <= 2).sum())
        note = (f"{len(plot_df)} of {len(tbl)} counties shown (blank = no drought year with a reported yield, "
                f"not zero loss). {thin} shown counties rest on 1-2 drought years, so treat them as noisy.")

    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin=dict(l=0, r=0, t=70, b=0), height=560)
    return fig, note

In [5]:
state_dd   = widgets.Dropdown(options=list(STATES),       value="Iowa",  description="State:",   layout=widgets.Layout(width="260px"))
crop_dd    = widgets.Dropdown(options=list(CROPS),        value="Corn (grain)", description="Crop:", layout=widgets.Layout(width="260px"))
def_dd     = widgets.Dropdown(options=list(DROUGHT_DEFS), value="D2+ severe or worse (severity ≥ 3)",
                              description="Drought:", layout=widgets.Layout(width="420px"))
measure_dd = widgets.Dropdown(options=[(k, v) for k, v in MEASURES.items()], value="exposure",
                              description="Map shows:", layout=widgets.Layout(width="420px"))
out = widgets.Output()

def refresh(_=None):
    with out:
        out.clear_output(wait=True)
        fig, note = make_map(state_dd.value, crop_dd.value, def_dd.value, measure_dd.value)
        if fig is not None:
            fig.show()
        print(note)
        print("Historical exposure, 2000–2025. Not a forecast; not a risk estimate.")

for w in (state_dd, crop_dd, def_dd, measure_dd):
    w.observe(refresh, names="value")

display(widgets.VBox([widgets.HBox([state_dd, crop_dd]), widgets.HBox([def_dd, measure_dd]), out]))
refresh()